# A-level English: pooling Language and Literature in one multilevel model

A-level English is really two subjects, English Language and English Literature, and a school's teaching in the two is likely related. Here we model both together against GCSE English value added (`P8MEAENG`).

The multilevel structure has three parts:

- a **school-level GCSE English VA** (latent, measured with known error);
- **subject-specific** intercept and slope on that GCSE VA, so Language and Literature can respond differently;
- a **shared school effect** that lifts or lowers a school's VA in *both* English A-levels. This is the pooling: a school with strong Literature results tells us something about its Language results, and vice versa, and schools with only one of the two subjects still contribute.

We compare it with a model where the two subjects are treated as independent. The combined A-level "English Language & Literature" is a separate qualification and is not included.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

print(f"Running on PyMC v{pm.__version__}")

## Data

We reshape `all-value-add-errors.csv` to long format: one row per school × subject, keeping the published confidence interval and entries. Schools need GCSE English VA and at least one of the two A-level VA scores. The standard error is `(upper - lower) / (2 * 1.96)`.

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

subjects = {"Language": "A-level English Language", "Literature": "A-level English Literature"}
school_cols = ["URN", "P8MEAENG", "P8MEAENG lower", "P8MEAENG upper", "P8 pupils", "RGN24NM"]

schools = (
    raw[school_cols]
    .rename(columns={"P8MEAENG": "gcse_va", "P8MEAENG lower": "gcse_lower", "P8MEAENG upper": "gcse_upper",
                     "P8 pupils": "p8_pupils", "RGN24NM": "region"})
    .dropna()
)

frames = []
for subject, prefix in subjects.items():
    sub = raw[["URN", f"{prefix} VA", f"{prefix} VA lower", f"{prefix} VA upper", f"{prefix} entries"]].dropna()
    sub.columns = ["URN", "alevel_va", "alevel_lower", "alevel_upper", "alevel_entries"]
    sub["subject"] = subject
    frames.append(sub)
long = pd.concat(frames)

Z_95 = 1.96
long["alevel_se"] = (long["alevel_upper"] - long["alevel_lower"]) / (2 * Z_95)
schools["gcse_se"] = (schools["gcse_upper"] - schools["gcse_lower"]) / (2 * Z_95)

long = long.merge(schools[["URN"]], on="URN", how="inner")
schools = schools[schools["URN"].isin(long["URN"])].reset_index(drop=True)

school_idx = pd.Series(np.arange(len(schools)), index=schools["URN"])
long["school_idx"] = long["URN"].map(school_idx).to_numpy()
long["subject_idx"] = long["subject"].map({"Language": 0, "Literature": 1}).to_numpy()
long = long.reset_index(drop=True)

n_per_school = long.groupby("URN")["subject"].nunique()
print(f"{len(schools)} schools, {len(long)} school-subject observations")
print(long["subject"].value_counts().to_string())
print(f"schools with both subjects: {(n_per_school == 2).sum()}, Language only: "
      f"{((n_per_school == 1) & long.groupby('URN')['subject'].first().eq('Language')).sum()}, "
      f"Literature only: {((n_per_school == 1) & long.groupby('URN')['subject'].first().eq('Literature')).sum()}")
long.groupby("subject")[["alevel_va", "alevel_se", "alevel_entries"]].describe().T

### Checking the confidence-interval formula

DfE's interval is $\pm 1.96\,\sigma_{national}/\sqrt{n}$, so $se\sqrt{n}$ should be constant within each subject.

In [ ]:
print((long["alevel_se"] * np.sqrt(long["alevel_entries"])).groupby(long["subject"]).agg(["mean", "std"]).round(3))
print("GCSE English:", (schools["gcse_se"] * np.sqrt(schools["p8_pupils"])).agg(["mean", "std"]).round(3).to_dict())

Each subject has its own near-constant national SD, so the standard errors are pure sampling noise.

## Exploratory look

Left: each subject against GCSE English VA. Right: for the schools that have both, Language against Literature.

In [ ]:
plot_df = long.merge(schools[["URN", "gcse_va", "gcse_se"]], on="URN")
colors = {"Language": "#DD8452", "Literature": "#4C72B0"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for subject, g in plot_df.groupby("subject"):
    axes[0].errorbar(g["gcse_va"], g["alevel_va"], xerr=g["gcse_se"], yerr=g["alevel_se"], fmt="o",
                     color=colors[subject], ecolor=colors[subject], alpha=0.2, markersize=3, elinewidth=0.5, label=subject)
axes[0].axhline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("GCSE English value-added (P8MEAENG)")
axes[0].set_ylabel("A-level value-added")
axes[0].legend()

wide = long.pivot(index="URN", columns="subject", values="alevel_va").dropna()
axes[1].scatter(wide["Literature"], wide["Language"], color="#4C72B0", alpha=0.3, s=14)
axes[1].axhline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("A-level English Literature VA")
axes[1].set_ylabel("A-level English Language VA")
axes[1].set_title(f"Schools with both (n={len(wide)}); raw correlation {wide['Literature'].corr(wide['Language']):.2f}")
plt.tight_layout()
plt.show()

The raw Language-Literature correlation is low, but each score is noisy (small cohorts), so the raw correlation understates how related the schools' *true* effects are. The model estimates that directly.

## Models

For school $i$ with true GCSE VA $x_i$, and subject $s \in \{\text{Language}, \text{Literature}\}$:

$$
\begin{aligned}
x_i &\sim \text{Normal}(\mu_x, \tau_x), \qquad x_i^{obs} \sim \text{Normal}(x_i, \sigma^x_i) \\
u_i &\sim \text{Normal}(0, \sigma_u) \qquad \text{(shared school effect)} \\
y_{is} &= \alpha_s + \beta_s x_i + u_i + \epsilon_{is}, \quad \epsilon_{is} \sim \text{Normal}(0, \tau_s), \qquad y_{is}^{obs} \sim \text{Normal}(y_{is}, \sigma^y_{is})
\end{aligned}
$$

The **separate** model is the same with $u_i = 0$: the two subjects share nothing beyond the GCSE score. Only two subjects means there is little information to estimate a spread of slopes across subjects, so each subject's $\alpha_s, \beta_s$ gets its own weakly informative prior rather than a hierarchical one. All effects are non-centered.

If $\sigma_u$ is large relative to $\tau_s$, a school's Language and Literature true effects move together. Their correlation, after removing the GCSE effect, is $\sigma_u^2 / \sqrt{(\sigma_u^2+\tau_L^2)(\sigma_u^2+\tau_T^2)}$.

In [ ]:
n_schools = len(schools)
gcse_obs = schools["gcse_va"].to_numpy()
gcse_se = schools["gcse_se"].to_numpy()
y_obs = long["alevel_va"].to_numpy()
y_se = long["alevel_se"].to_numpy()
s_idx = long["school_idx"].to_numpy()
subj_idx = long["subject_idx"].to_numpy()
n_obs = len(long)

def build(shared_school_effect):
    with pm.Model(coords={"subject": ["Language", "Literature"]}) as m:
        mu_x = pm.Normal("mu_x", 0, 1)
        tau_x = pm.HalfNormal("tau_x", 1)
        x_raw = pm.Normal("x_raw", 0, 1, shape=n_schools)
        x_true = pm.Deterministic("x_true", mu_x + tau_x * x_raw)

        alpha = pm.Normal("alpha", 0, 1, dims="subject")
        beta = pm.Normal("beta", 0, 1, dims="subject")
        tau_y = pm.HalfNormal("tau_y", 1, dims="subject")
        e_raw = pm.Normal("e_raw", 0, 1, shape=n_obs)

        mean = alpha[subj_idx] + beta[subj_idx] * x_true[s_idx]
        if shared_school_effect:
            sigma_u = pm.HalfNormal("sigma_u", 0.5)
            u_raw = pm.Normal("u_raw", 0, 1, shape=n_schools)
            u = pm.Deterministic("u", sigma_u * u_raw)
            mean = mean + u[s_idx]
        y_true = pm.Deterministic("y_true", mean + tau_y[subj_idx] * e_raw)

        pm.Normal("gcse_obs", mu=x_true, sigma=gcse_se, observed=gcse_obs)
        pm.Normal("alevel_obs", mu=y_true, sigma=y_se, observed=y_obs)
    return m

pooled_model = build(shared_school_effect=True)
separate_model = build(shared_school_effect=False)

### Prior predictive check

In [ ]:
with pooled_model:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(prior.prior_predictive["gcse_obs"].to_numpy().ravel(), bins=40, color="#4C72B0")
axes[0].set_title("Prior predictive: GCSE English VA")
axes[1].hist(prior.prior_predictive["alevel_obs"].to_numpy().ravel(), bins=40, color="#4C72B0")
axes[1].set_title("Prior predictive: A-level VA")
plt.tight_layout()
plt.show()

### Fit

In [ ]:
sample_kwargs = dict(draws=1000, tune=2000, chains=4, target_accept=0.99, random_seed=RANDOM_SEED, progressbar=False)

with pooled_model:
    idata_pooled = pm.sample(**sample_kwargs)
    idata_pooled.update(pm.sample_posterior_predictive(idata_pooled, random_seed=RANDOM_SEED))

with separate_model:
    idata_separate = pm.sample(**sample_kwargs)

### Diagnostics

In [ ]:
for name, idt in [("pooled", idata_pooled), ("separate", idata_separate)]:
    print(f"{name}: divergences = {int(idt.sample_stats['diverging'].sum())}")

az.summary(idata_pooled, var_names=["alpha", "beta", "tau_y", "sigma_u", "mu_x", "tau_x"], round_to=3)

In [ ]:
az.summary(idata_separate, var_names=["alpha", "beta", "tau_y"], round_to=3)

## What the pooling shows

### How related are a school's Language and Literature effects?

The correlation between the two subjects' true school effects, after removing the GCSE effect, and the share of each subject's residual variance that is a shared school effect.

In [ ]:
post = idata_pooled.posterior
sigma_u = post["sigma_u"].to_numpy().ravel()
tau_l = post["tau_y"].sel(subject="Language").to_numpy().ravel()
tau_t = post["tau_y"].sel(subject="Literature").to_numpy().ravel()

rho = sigma_u**2 / np.sqrt((sigma_u**2 + tau_l**2) * (sigma_u**2 + tau_t**2))
shared_lang = sigma_u**2 / (sigma_u**2 + tau_l**2)
shared_lit = sigma_u**2 / (sigma_u**2 + tau_t**2)

def summarise(x):
    lo, hi = np.percentile(x, [5.5, 94.5])
    return f"mean={x.mean():.3f}, 89% interval [{lo:.3f}, {hi:.3f}]"

print("sigma_u (shared school effect SD):", summarise(sigma_u))
print("Language-Literature correlation of school effects:", summarise(rho))
print("Share of Language residual variance that is shared:", summarise(shared_lang))
print("Share of Literature residual variance that is shared:", summarise(shared_lit))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(rho, bins=40, density=True, color="#55A868")
ax.set_xlabel("Correlation of Language and Literature school effects")
plt.show()

### Slopes by subject

In [ ]:
beta_pooled = idata_pooled.posterior["beta"]
beta_separate = idata_separate.posterior["beta"]

fig, ax = plt.subplots(figsize=(8, 4.5))
for subject in ["Language", "Literature"]:
    ax.hist(beta_pooled.sel(subject=subject).to_numpy().ravel(), bins=40, density=True, alpha=0.5,
            color=colors[subject], label=f"{subject} (pooled model)")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("beta: A-level VA per unit of GCSE English VA")
ax.legend()
plt.show()

diff = (beta_pooled.sel(subject="Literature") - beta_pooled.sel(subject="Language")).to_numpy().ravel()
print("beta, Literature minus Language (pooled):", summarise(diff))
for name, b in [("pooled", beta_pooled), ("separate", beta_separate)]:
    for subject in ["Language", "Literature"]:
        print(f"beta {subject:10s} {name:8s}", summarise(b.sel(subject=subject).to_numpy().ravel()))

## Goodness of fit

Bayesian $R^2$ on the latent scale, per subject. The structural part is $\alpha_s + \beta_s x_i$; in the pooled model we also report the fit including the shared school effect $u_i$, which is variation explained by school rather than by GCSE. The residual variance is $\tau_s^2$ (separate) or $\tau_s^2$ after $u$ (pooled).

In [ ]:
def r2_by_subject(idata, pooled):
    p = idata.posterior
    x = p["x_true"].to_numpy()  # chain, draw, school
    out = {}
    for s, name in enumerate(["Language", "Literature"]):
        rows = np.where(subj_idx == s)[0]
        sch = s_idx[rows]
        a = p["alpha"].sel(subject=name).to_numpy()[..., None]
        b = p["beta"].sel(subject=name).to_numpy()[..., None]
        tau = p["tau_y"].sel(subject=name).to_numpy()
        gcse_fit = a + b * x[..., sch]
        var_gcse = gcse_fit.var(axis=-1)
        if pooled:
            var_u = p["u"].to_numpy()[..., sch].var(axis=-1)
            total_fit = (a + b * x[..., sch] + p["u"].to_numpy()[..., sch]).var(axis=-1)
            out[name] = ((var_gcse / (var_gcse + var_u + tau**2)).ravel(),
                         (total_fit / (total_fit + tau**2)).ravel())
        else:
            out[name] = ((var_gcse / (var_gcse + tau**2)).ravel(), None)
    return out

r2_sep = r2_by_subject(idata_separate, pooled=False)
r2_pool = r2_by_subject(idata_pooled, pooled=True)

for name in ["Language", "Literature"]:
    print(f"{name}: R² from GCSE only, separate model:        {summarise(r2_sep[name][0])}")
    print(f"{name}: R² from GCSE only, pooled model:          {summarise(r2_pool[name][0])}")
    print(f"{name}: R² from GCSE + shared school effect:      {summarise(r2_pool[name][1])}")

### Residual vs. GCSE value-added

Standardised residuals from the pooled model, binned over GCSE English VA, for each subject. Flat near zero means the linear relationship is adequate.

In [ ]:
pred = idata_pooled.posterior_predictive["alevel_obs"].to_numpy().reshape(-1, n_obs)
long["std_resid"] = (y_obs - pred.mean(axis=0)) / pred.std(axis=0)
long = long.merge(schools[["URN", "gcse_va"]], on="URN", how="left")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, subject in zip(axes, ["Language", "Literature"]):
    g = long[long["subject"] == subject]
    binned = g.groupby(pd.qcut(g["gcse_va"], 12), observed=True).agg(gcse_va=("gcse_va", "mean"), r=("std_resid", "mean"))
    ax.scatter(g["gcse_va"], g["std_resid"], color=colors[subject], alpha=0.2, s=12)
    ax.plot(binned["gcse_va"], binned["r"], color="black", marker="o", markersize=4, linewidth=1.2, label="binned mean (12 bins)")
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    ax.set_title(subject)
    ax.set_xlabel("GCSE English VA (P8MEAENG)")
axes[0].set_ylabel("Standardised residual")
axes[0].legend()
plt.tight_layout()
plt.show()

### Do the strongest GCSE schools "top out"?

Mean A-level VA by GCSE English VA quintile, for each subject.

In [ ]:
quint = []
for subject, g in long.groupby("subject"):
    t = g.groupby(pd.qcut(g["gcse_va"], 5, labels=range(1, 6)), observed=True).agg(
        mean_gcse_va=("gcse_va", "mean"), mean_alevel_va=("alevel_va", "mean"), n=("URN", "count"))
    t["subject"] = subject
    quint.append(t)
quintile_table = pd.concat(quint).reset_index().rename(columns={"gcse_va": "gcse_quintile"})
display(quintile_table.pivot(index="gcse_quintile", columns="subject", values=["mean_gcse_va", "mean_alevel_va", "n"]).round(3))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
width = 0.38
for k, subject in enumerate(["Language", "Literature"]):
    t = quintile_table[quintile_table["subject"] == subject]
    ax.bar(t["gcse_quintile"].astype(int) + (k - 0.5) * width, t["mean_alevel_va"], width, color=colors[subject], label=subject)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("GCSE English VA quintile (1 = lowest, 5 = highest)")
ax.set_ylabel("Mean A-level VA")
ax.legend()
plt.show()

## Interpretation

*(filled in after inspecting the results)*